# 📚 StuLearn RAG Pipeline using LlamaIndex

## Overview

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline for **StuLearn**, an AI-powered study assistant that answers questions using uploaded academic documents.

The pipeline combines:

- **LlamaIndex** for document indexing and retrieval
- **ChromaDB** as the vector database
- **OpenRouter** for embedding generation and LLM inference
- **Custom Prompt Engineering** to minimize hallucinations and generate grounded responses

---

## Pipeline Architecture


Documents (PDFs)
        │
        ▼
Load Documents
        │
        ▼
Sentence Chunking
        │
        ▼
Generate Embeddings
        │
        ▼
Store in ChromaDB
        │
        ▼
Retrieve Relevant Chunks
        │
        ▼
Custom Prompt
        │
        ▼
OpenRouter LLM
        │
        ▼
Grounded Answer


---

## Objectives

- Load research papers from a local directory
- Split documents into meaningful chunks
- Generate semantic embeddings
- Store embeddings in a persistent vector database
- Retrieve the most relevant chunks for a user query
- Generate accurate answers strictly based on retrieved context

---

**Author:** Hrishikesh Sanap

**Project:** Student Learn – AI-Powered Study Assistant

Imports



In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import os

from dotenv import load_dotenv

# ---------- LlamaIndex ----------
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    PromptTemplate,
)

from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai_like import OpenAILike

# ---------- Chroma ----------
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

In [ ]:
# ==========================================================
# Load Environment Variables
# ==========================================================

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

In [ ]:
# ==========================================================
# Project Configuration
# ==========================================================

DATA_DIRECTORY = "../data"

CHROMA_DB_PATH = "../storage/chroma_db"

COLLECTION_NAME = "student_notes"

EMBEDDING_MODEL = "text-embedding-3-large"

LLM_MODEL = "openrouter/free"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

TOP_K = 5

In [ ]:
# ==========================================================
# Load Documents
# ==========================================================

documents = SimpleDirectoryReader(
    input_dir=DATA_DIRECTORY
).load_data()

print(f"Loaded {len(documents)} document(s).")

In [ ]:
# ==========================================================
# Split Documents into Chunks
# ==========================================================

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print(f"Generated {len(nodes)} text chunks.")

In [ ]:
# ==========================================================
# Configure Embedding Model
# ==========================================================

embed_model = OpenAIEmbedding(
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    model=EMBEDDING_MODEL,
)

Settings.embed_model = embed_model

In [ ]:
# ==========================================================
# Create Persistent Chroma Database
# ==========================================================

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

In [ ]:
# ==========================================================
# Create Persistent Chroma Database
# ==========================================================

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

In [ ]:
# ==========================================================
# Build Vector Index
# ==========================================================

index = VectorStoreIndex(
    nodes,
    vector_store=vector_store,
    embed_model=embed_model,
)

print("Vector index created successfully.")

In [ ]:
# ==========================================================
# Configure Retriever
# ==========================================================

retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

In [ ]:
# ==========================================================
# Test Retrieval
# ==========================================================

query = "What is performance evaluation?"

results = retriever.retrieve(query)

print(f"Retrieved {len(results)} chunks\n")

for i, node in enumerate(results, start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(node.text[:500])
    print()

In [ ]:
# ==========================================================
# Configure LLM
# ==========================================================

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    is_chat_model=True,
    timeout=300,
)

In [ ]:
from llama_index.core import PromptTemplate

rag_prompt = PromptTemplate("""
You are StuLearn, an AI-powered research and study assistant.

Your task is to answer questions ONLY using the provided context.

Rules:
1. Never use outside knowledge.
2. If the answer is not present, reply:
   "I could not find this information in the uploaded documents."
3. Combine information from multiple retrieved chunks whenever necessary.
4. Give detailed, well-structured answers.
5. Use headings and bullet points where appropriate.
6. Explain technical terms in simple language.
7. Do not hallucinate or guess.
8. If multiple uploaded papers discuss the topic, mention that the answer combines information from multiple papers.

-----------------------
Context
-----------------------
{context_str}

-----------------------
Question
-----------------------
{query_str}

Provide a comprehensive answer:
""")

In [ ]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=TOP_K,
    text_qa_template=rag_prompt,
    streaming=True,
)

In [ ]:
query = "What role does machine learning play in the system?"

retrieved_nodes = retriever.retrieve(query)

print(f"Retrieved {len(retrieved_nodes)} chunks\n")

for i, node in enumerate(retrieved_nodes, start=1):
    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Score : {node.score:.4f}")
    print("-" * 80)
    print(node.text[:600])
    print()

In [ ]:
context = "\n\n".join(
    [node.text for node in retrieved_nodes]
)

print(context)

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

context_tokens = len(
    encoding.encode(context)
)

print(f"Context Tokens : {context_tokens}")

In [ ]:
query_tokens = len(
    encoding.encode(query)
)

print(f"Query Tokens : {query_tokens}")

In [ ]:
full_prompt = f"""
Context

{context}

Question

{query}
"""

prompt_tokens = len(
    encoding.encode(full_prompt)
)

print(prompt_tokens)

In [ ]:
response = query_engine.query(query)

answer = str(response)

answer_tokens = len(
    encoding.encode(answer)
)

print(answer_tokens)

In [ ]:
print("=" * 60)

print(f"Query Tokens     : {query_tokens}")
print(f"Context Tokens   : {context_tokens}")
print(f"Answer Tokens    : {answer_tokens}")

print(f"Total Tokens     : {query_tokens + context_tokens + answer_tokens}")

In [ ]:
import time

start = time.perf_counter()

results = retriever.retrieve(query)

retrieval_time = time.perf_counter() - start

print(f"Retrieval Time : {retrieval_time:.3f} sec")

In [ ]:
start = time.perf_counter()

response = query_engine.query(query)

generation_time = time.perf_counter() - start

print(f"Generation Time : {generation_time:.3f} sec")

============================================================
RAG Evaluation Summary
============================================================

Query
------------------------------------------------------------
What role does machine learning play in the system?

Retrieved Chunks     : 5
Context Tokens       : 1867
Question Tokens      : 11
Answer Tokens        : 452
Total Tokens         : 2330

Retrieval Time       : 0.09 sec
Generation Time      : 1.74 sec

Embedding Model      : text-embedding-3-large
LLM                  : openrouter/free

============================================================